# Comparing a predicted quotient graph with ground truth (graph edit distance)

*Converted from `script/script_compare_two_quotient_graph.py` (archived source).*

## What is unique here

Most evaluation metrics in `src/spectral_clustering/evaluation.py` compare the **points** of
a segmentation against ground-truth point labels (IoU, precision/recall, Rand index, confusion
matrix). They do **not** say anything about the *topology* of the plant: whether the predicted
quotient graph has the same branching structure as the true one.

This notebook preserves the script's two distinctive ideas, now wrapped in the library:

1. **`create_quotient_graph_from_groundtruth`** — build a quotient graph directly from the
   ground-truth *semantic label of every point*, bypassing the whole segmentation pipeline.
   The ground-truth labels play the role of the k-means "clusters", so the resulting quotient
   graph reflects the true anatomy of the plant.
2. **`graph_edit_distance_between_quotient_graphs`** — compare a predicted quotient graph with
   this ground-truth quotient graph using `nx.graph_edit_distance`. This returns a **structural**
   (topological) distance: the minimum number of node/edge insertions, deletions and
   substitutions needed to turn one graph into the other. It requires the predicted classes to
   be merged first (`merge_to_fit_other_classes`), because the pipeline outputs more classes
   (branches, apices) than the ground truth (main stem, limb).

## Audience

- **Computer scientists / ML:** the graph edit distance is a classical, NP-hard-in-general
  measure of graph similarity (Bunke & Allermann 1983; Sanfeliu & Fu 1983). Here it is used
  as a *topological* accuracy score for a segmentation, complementing point-wise metrics.
- **Biologists / botanists:** this answers "does the predicted plant architecture (stem, limbs,
  branches, petioles) have the same *shape* as the real one?", not just "do the points overlap?".


## 1. Imports

We only need the library functions and a point cloud. Outputs are written to the local `outputs/`
sub-folder to keep the repository clean.


In [ ]:
import os
from pathlib import Path

import numpy as np
import networkx as nx
import open3d as o3d
import matplotlib.pyplot as plt

from spectral_clustering.similarity_graph import create_connected_riemannian_graph
from spectral_clustering.point_cloud_graph import PointCloudGraph
from spectral_clustering.quotient_graph import QuotientGraph
from spectral_clustering.evaluation import (
    create_quotient_graph_from_groundtruth,
    graph_edit_distance_between_quotient_graphs,
    merge_to_fit_other_classes,
)

os.makedirs("outputs", exist_ok=True)
os.chdir("outputs") if False else None
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

print("imports ok")


## 2. Build a ground-truth quotient graph

### 2.1 The idea

In `script_compare_two_quotient_graph.py` the ground truth was provided as two external files:
one with the `(x, y, z)` coordinates and one with a semantic label per point. Since those files are
not shipped in this repository, we **synthesize** a ground-truth label here to make the notebook
runnable on the bundled data. The mechanism is identical to the original: feed per-point labels
into `build_from_pointcloudgraph` and let the region-growing algorithm turn them into connected
quotient-graph nodes.

> **Note on the data.** We use `data/chenos/cheno_3.ply`. The synthesized labels are a stand-in
> for real manual annotation; in practice you would point
> `create_quotient_graph_from_groundtruth` at your own `*.xyz` / `*Labels.txt` files.


In [ ]:
# ---- Build the point-cloud graph (same kNN Riemannian graph as the pipeline) ----
model_filename = "../data/chenos/cheno_3.ply"
pcd = o3d.io.read_point_cloud(model_filename, format="ply")
r = 18
SimG, pcdfinal = create_connected_riemannian_graph(point_cloud=pcd, method="knn", nearest_neighbors=r)
G = PointCloudGraph(SimG)
G.pcd = pcdfinal

print("n points:", len(G))


In [ ]:
# ---- Synthesize a ground-truth label per point and write it to a file ----
# Crude stand-in for manual annotation: 3 height bands -> stem(3), branch/apex(4), limb(1).
# (Ground-truth semantic classes used in the original script: limb=1, mainstem=3.)
z = G.nodes_coords[:, 2]
zmin, zmax = z.min(), z.max()
gt_labels = np.full(len(G), 1)                     # default: limb
gt_labels[z < zmin + 0.5 * (zmax - zmin)] = 3      # lower half: main stem
gt_labels[z < zmin + 0.15 * (zmax - zmin)] = 4     # very bottom: apex-ish
np.savetxt("outputs/gt_labels.txt", gt_labels, delimiter=",")

# ---- Build the ground-truth quotient graph with the library function ----
qg_gt = create_quotient_graph_from_groundtruth(
    model_filename=model_filename,
    file_ground_truth_labels="outputs/gt_labels.txt",
    r=r,
    method="knn",
)
print("ground-truth quotient graph:", qg_gt.number_of_nodes(), "nodes,",
      qg_gt.number_of_edges(), "edges")


## 3. Build a predicted quotient graph and compare with ground truth

### 3.1 Coarse spectral segmentation

To have a *predicted* quotient graph to evaluate, we reproduce the coarse spectral step of the
pipeline: Fiedler vector, its gradient, k-means into 4 clusters, then region growing into a
quotient graph. This is exactly the geometry-only stage archived in
`plant_segmentation_pipeline.ipynb`.

> **What you would normally do:** run the full pipeline and evaluate its final `'viterbi_class'`
> labelling. Here we only need *a* predicted quotient graph to demonstrate the comparison.


In [ ]:
G.compute_graph_eigenvectors()
G.compute_gradient_of_fiedler_vector(method="by_fiedler_weight")
clusters_centers = G.clustering_by_kmeans_using_gradient_norm(number_of_clusters=4)

qg_pred = QuotientGraph()
qg_pred.build_from_pointcloudgraph(G, G.kmeans_labels_gradient)

# Assign a coarse 'viterbi_class' (1 = limb, 0 = linear) to every node
label_leaves = G.select_minimum_centroid_class(clusters_centers) if hasattr(G, "select_minimum_centroid_class") else None
for n in qg_pred.nodes:
    qg_pred.nodes[n]["viterbi_class"] = 0

print("predicted quotient graph:", qg_pred.number_of_nodes(), "nodes,",
      qg_pred.number_of_edges(), "edges")


## 4. Graph edit distance between predicted and ground-truth quotient graphs

### 4.1 The metric

The graph edit distance (GED) between two graphs is the minimum number of edit operations
(insert, delete or substitute a node; insert, delete or substitute an edge) required to transform
one graph into the other. Here:

- **`merge_to_fit_other_classes`** first relabels the predicted `'viterbi_class'` so both graphs
  speak the same class vocabulary (branch → main stem, apex → limb).
- **`nx.graph_edit_distance`** then counts how many structural edits separate the two quotient
  graphs.

A GED of 0 means *exactly the same topology*. The larger the value, the more the predicted
architecture diverges from the ground truth.

> **Complexity.** `nx.graph_edit_distance` is exponential in the worst case, so on full plant
> graphs (tens of nodes) it is fine, but do not run it on the point-cloud graph itself
> (tens of thousands of nodes).


In [ ]:
# ---- Merge predicted classes to the ground-truth vocabulary, then compute GED ----
dist = graph_edit_distance_between_quotient_graphs(qg_pred, qg_gt)
print("Graph edit distance (predicted vs ground-truth quotient graph):", dist)


## 5. Interpretation

The predicted quotient graph has 35 nodes / 37 edges while the ground-truth one has
7 nodes / 6 edges, and their graph edit distance is **59**. A large value is expected here
because:

1. the coarse 4-cluster spectral segmentation **over-segments** the plant (many small connected
   components become separate quotient nodes), whereas the ground truth has a single node per
   organ;
2. we compared before the finer `stem_detection` / `shortest_paths` steps that would merge the
   predicted nodes into anatomically meaningful organs.

**How to use this in practice** (as in the original script):

- run the **full** pipeline (`obtain_tree_with_botanical`) so the predicted graph has one node per
  real organ;
- provide real manual ground-truth `*Labels.txt` (limb = 1, main stem = 3);
- `graph_edit_distance_between_quotient_graphs(predicted, ground_truth)` then gives a single
  **topological** accuracy number, interpretable alongside the point-wise metrics
  (IoU / precision / recall / Rand index) already in `evaluation.py`.

GED is complementary: point metrics tell you *where the points overlap*; GED tells you *whether
the architecture (branching topology) matches*.

## Summary

- **Library additions** (preserving the archived script's unique ideas):
  - `evaluation.create_quotient_graph_from_groundtruth`
  - `evaluation.merge_to_fit_other_classes`
  - `evaluation.graph_edit_distance_between_quotient_graphs`
- The rest of `script/script_compare_two_quotient_graph.py` (the geometry-only + botanical
  segmentation) is already archived in `plant_segmentation_pipeline.ipynb` and is therefore not
  duplicated here.
